In [ ]:
import xml.etree.ElementTree as ET
import json
from pathlib import Path

# Set Iris Metadata

In [ ]:
# Metadata for Iris Dataset
iris_metadata = {
    "identifier": "10.24432/C56C76",
    "identifier_type": "DOI",
    "title": "Iris Species Dataset",
    "creator": "Ronald Aylmer Fisher",
    "description": "A multivariate data set introduced by the British statistician and biologist Ronald Fisher in his 1936 paper.",
    "publisher": "UCI Machine Learning Repository",
    "publication_year": "1936",
    "url": "https://archive.ics.uci.edu/dataset/53/iris"
}

def local_name(tag):
    if '}' in tag:
        return tag.split('}', 1)[1]
    return tag

# Generate XML file

In [ ]:
def save_datacite_xml(meta, filename):
    # Create root with namespace
    root = ET.Element("resource", xmlns="http://datacite.org/schema/kernel-4")
    
    # Identifier
    ET.SubElement(root, "identifier", identifierType=meta["identifier_type"]).text = meta["identifier"]
    
    # Creator
    creators = ET.SubElement(root, "creators")
    creator = ET.SubElement(creators, "creator")
    ET.SubElement(creator, "creatorName").text = meta["creator"]
    
    # Title
    titles = ET.SubElement(root, "titles")
    ET.SubElement(titles, "title").text = meta["title"]
    
    # Publisher
    ET.SubElement(root, "publisher").text = meta["publisher"]

    # Publication Year
    ET.SubElement(root, "publicationYear").text = meta["publication_year"]

    # Resource Type
    ET.SubElement(root, "resourceType", resourceTypeGeneral="Dataset").text = "Dataset"
    
    # Description
    descriptions = ET.SubElement(root, "descriptions")
    ET.SubElement(descriptions, "description", descriptionType="Abstract").text = meta["description"]
    
    # Save the file
    tree = ET.ElementTree(root)
    tree.write(filename, encoding="utf-8", xml_declaration=True)
    print(f"Success: DataCite XML saved to {filename}")

def parse_datacite_xml(xml_file: Path) -> dict:
    """Parses the XML file step by step as defined in the notebook."""
    root = ET.parse(xml_file).getroot()

    identifier = None
    creators = []
    title = None
    publisher = None
    publication_year = None
    resource_type = None
    description = None

    for el in root.iter():
        name = local_name(el.tag)
        text = (el.text or "").strip()

        if name == "identifier" and text and identifier is None:
            identifier = text
        elif name == "creatorName" and text:
            creators.append(text)
        elif name == "title" and text and title is None:
            title = text
        elif name == "publisher" and text and publisher is None:
            publisher = text
        elif name == "publicationYear" and text and publication_year is None:
            publication_year = text
        elif name == "resourceType" and text and resource_type is None:
            resource_type = text
        elif name == "description" and text and description is None:
            description = text

    return {
        "identifier": identifier,
        "creators": creators,
        "title": title,
        "publisher": publisher,
        "publicationYear": publication_year,
        "resourceType": resource_type,
        "description": description,
    }

# Generate JSON file

In [ ]:
def save_schema_jsonld(parsed_meta, filename, original_url):
    schema_dict = {
        "@context": "https://schema.org/",
        "@type": "Dataset",
        "name": parsed_meta["title"],
        "description": parsed_meta["description"],
        "creator": {
            "@type": "Person",
            "name": parsed_meta["creators"][0] if parsed_meta["creators"] else "Unknown"
        },
        "publisher": {
            "@type": "Organization",
            "name": parsed_meta["publisher"]
        },
        "datePublished": parsed_meta["publicationYear"],
        "url": original_url,
        "identifier": parsed_meta["identifier"]
    }
    
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(schema_dict, f, indent=4)
    print(f"Success: Schema.org JSON-LD saved to {filename}")

In [ ]:
raw_xml = xml_path.read_text(encoding="utf-8")
print(raw_xml[:1500])

<?xml version='1.0' encoding='utf-8'?>
<resource xmlns="http://datacite.org/schema/kernel-4"><identifier identifierType="DOI">10.24432/C56C76</identifier><creators><creator><creatorName>Ronald Aylmer Fisher</creatorName></creator></creators><titles><title>Iris Species Dataset</title></titles><publisher>UCI Machine Learning Repository</publisher><publicationYear>1936</publicationYear><resourceType resourceTypeGeneral="Dataset">Dataset</resourceType><descriptions><description descriptionType="Abstract">A multivariate data set introduced by the British statistician and biologist Ronald Fisher in his 1936 paper.</description></descriptions></resource>


In [ ]:
raw_text = jsonld_output 

obj = json.loads(raw_text)
print("Python object type:", type(obj))

if isinstance(obj, dict):
    print("Top-level keys:", list(obj.keys()))
    
    if "@context" in obj:
        print("Successfully validated as a JSON-LD object.")
else:
    print("Top-level object is not a dictionary.")

Python object type: <class 'dict'>
Top-level keys: ['@context', '@type', 'name', 'description', 'creator', 'publisher', 'datePublished', 'url', 'identifier']
Successfully validated as a JSON-LD object.


# Save data to files

In [ ]:
if __name__ == "__main__":
    xml_path = Path("iris_datacite.xml")
    json_path = Path("iris_schemaorg.json")
    
    # 1. Create the XML file first
    save_datacite_xml(iris_metadata, xml_path)
    
    # 2. Use your custom parser to extract the data back out
    parsed_data = parse_datacite_xml(xml_path)
    
    # 3. Create the JSON-LD using that parsed data
    save_schema_jsonld(parsed_data, json_path, iris_metadata["url"])

Success: DataCite XML saved to iris_datacite.xml
Success: Schema.org JSON-LD saved to iris_schemaorg.json
